# 01 — Person detection: fine-tuning YOLO on VisDrone (aerial persons)
This notebook quantifies the aerial domain gap for person detection and closes
part of it by fine-tuning YOLO11s on the VisDrone person subset. It produces the
before-and-after AP50 comparison reported in the paper. The approximate runtime
is 40 minutes on a T4 GPU.


### Setup
The notebook requires a GPU runtime and the project source code.
1. Enable the GPU by selecting **Runtime → Change runtime type → T4 GPU**.
2. Archive the local `src/` and `scripts/` directories as `src.zip`
   (`cd Project && zip -r src.zip src scripts`), then either upload the archive
   in the cell below or place it in Drive and set `SRC_ZIP`.


In [1]:
# Environment setup.
!pip -q install ultralytics rtmlib onnxruntime-gpu
import torch, os
print('cuda:', torch.cuda.is_available())

# Project code: upload src.zip (or mount Drive and set SRC_ZIP).
from pathlib import Path
SRC_ZIP = None  # e.g. '/content/drive/MyDrive/sar_project/src.zip'
if SRC_ZIP is None:
    from google.colab import files
    up = files.upload()  # choose src.zip
    SRC_ZIP = next(iter(up))
!mkdir -p /content/project && unzip -q -o "$SRC_ZIP" -d /content/project
import sys
sys.path.insert(0, '/content/project/src')
sys.path.insert(0, '/content/project')
print('project code ready')

# Results are written to Drive so that they persist across the session.
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/sar_project_results'); OUT.mkdir(parents=True, exist_ok=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.8/249.8 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 40.5 MB/s eta 0:00:00
cuda: True


Saving src.zip to src.zip
project code ready
Mounted at /content/drive


In [2]:
# Download VisDrone-DET from the Ultralytics mirror.
%cd /content
!curl -sL -o vd_train.zip https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip
!curl -sL -o vd_val.zip   https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip
!mkdir -p data/raw && unzip -q vd_train.zip -d data/raw && unzip -q vd_val.zip -d data/raw && rm vd_*.zip


/content


In [3]:
# Convert the annotations to the YOLO person-only format using the project converter.
import config
from pathlib import Path
config.RAW_DIR = Path('/content/data/raw')
config.DATA_DIR = Path('/content/data')
config.VISDRONE_TRAIN = config.RAW_DIR / 'VisDrone2019-DET-train'
config.VISDRONE_VAL = config.RAW_DIR / 'VisDrone2019-DET-val'
config.VISDRONE_PERSON = config.DATA_DIR / 'visdrone_person'
from data import visdrone
visdrone.VISDRONE_PERSON = config.VISDRONE_PERSON
visdrone.DATA_DIR = config.DATA_DIR
visdrone.convert_split(config.VISDRONE_TRAIN, 'train', config.VISDRONE_PERSON)
visdrone.convert_split(config.VISDRONE_VAL, 'val', config.VISDRONE_PERSON)
yaml_path = visdrone.write_dataset_yaml(config.VISDRONE_PERSON)


train: 6471 images, 106396 person boxes -> /content/data/visdrone_person
val: 548 images, 13969 person boxes -> /content/data/visdrone_person
wrote /content/data/visdrone_person.yaml


In [4]:
# Evaluate the zero-shot COCO yolo11s model on aerial imagery as the baseline.
import eval_detect
eval_detect.VISDRONE_PERSON = config.VISDRONE_PERSON
base = eval_detect.evaluate_on_visdrone('yolo11s.pt', imgsz=1280,
                                        tag='yolo11s zero-shot (COCO)', device=0)


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
  50/548 images
  100/548 images
  150/548 images
  200/548 images
  250/548 images
  300/548 images
  350/548 images
  400/548 images
  450/548 images
  500/548 images
{'AP50': 0.4368, 'precision': 0.2473, 'recall': 0.5964, 'n_gt': 13969, 'n_pred': 33694, 'model': 'yolo11s zero-shot (COCO)', 'split': 'val', 'imgsz': 1280, 'n_images': 548}


In [5]:
# Fine-tune on the full training split; approximately 30 minutes on a T4 at imgsz 1280.
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
model.train(data=str(yaml_path), epochs=30, imgsz=1280, batch=8, device=0,
            project='/content/runs', name='yolo11s_visdrone_ft', exist_ok=True)
best = '/content/runs/yolo11s_visdrone_ft/weights/best.pt'


Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/visdrone_person.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11s_visdrone_ft, nbs=64

In [6]:
# Evaluate the fine-tuned model with the same evaluator and the same split.
ft = eval_detect.evaluate_on_visdrone(best, imgsz=1280,
                                      tag='yolo11s fine-tuned (30ep)', device=0)
import pandas as pd
df = pd.DataFrame([base, ft])
df.to_csv(OUT / 'detection_visdrone.csv', index=False)
!cp {best} {OUT}/yolo11s_visdrone_best.pt
df


  50/548 images
  100/548 images
  150/548 images
  200/548 images
  250/548 images
  300/548 images
  350/548 images
  400/548 images
  450/548 images
  500/548 images
{'AP50': 0.7086, 'precision': 0.1905, 'recall': 0.8586, 'n_gt': 13969, 'n_pred': 62955, 'model': 'yolo11s fine-tuned (30ep)', 'split': 'val', 'imgsz': 1280, 'n_images': 548}


,AP50,precision,recall,n_gt,n_pred,model,split,imgsz,n_images
0,0.436766,0.247255,0.596392,13969,33694,yolo11s zero-shot (COCO),val,1280,548
1,0.708611,0.190517,0.858616,13969,62955,yolo11s fine-tuned (30ep),val,1280,548


The increase in AP50 from the zero-shot model to the fine-tuned model is the
detection domain-gap result reported in Section 5.1. Varying the input size
(`--imgsz 640` against `1280`) provides a resolution ablation.


In [7]:
!zip -r /tmp/detection_outputs.zip \
    /content/runs \
    /content/project

  adding: content/runs/ (stored 0%)
  adding: content/runs/yolo11s_visdrone_ft/ (stored 0%)
  adding: content/runs/yolo11s_visdrone_ft/train_batch16180.jpg (deflated 12%)
  adding: content/runs/yolo11s_visdrone_ft/BoxF1_curve.png (deflated 17%)
  adding: content/runs/yolo11s_visdrone_ft/results.png (deflated 7%)
  adding: content/runs/yolo11s_visdrone_ft/labels.jpg (deflated 36%)
  adding: content/runs/yolo11s_visdrone_ft/train_batch2.jpg (deflated 2%)
  adding: content/runs/yolo11s_visdrone_ft/val_batch2_pred.jpg (deflated 0%)
  adding: content/runs/yolo11s_visdrone_ft/train_batch0.jpg (deflated 6%)
  adding: content/runs/yolo11s_visdrone_ft/train_batch1.jpg (deflated 4%)
  adding: content/runs/yolo11s_visdrone_ft/val_batch0_labels.jpg (deflated 1%)
  adding: content/runs/yolo11s_visdrone_ft/results.csv (deflated 60%)
  adding: content/runs/yolo11s_visdrone_ft/args.yaml (deflated 53%)
  adding: content/runs/yolo11s_visdrone_ft/train_batch16181.jpg (deflated 13%)
  adding: content/runs